# Download Dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("PromptCloudHQ/amazon-reviews-unlocked-mobile-phones")

print("Path to dataset files:", path)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 32.6M/32.6M [00:06<00:00, 5.31MB/s]

Extracting files...


Path to dataset files: C:\Users\Ahmed\.cache\kagglehub\datasets\PromptCloudHQ\amazon-reviews-unlocked-mobile-phones\versions\1


# Task 1: Load the Data and Perform 5 Preprocessing Steps

In [2]:
import re
import pandas as pd

# Path to the downloaded dataset from Kaggle
path = r"C:\Users\Ahmed\.cache\kagglehub\datasets\PromptCloudHQ\amazon-reviews-unlocked-mobile-phones\versions\1\Amazon_Unlocked_Mobile.csv"
df = pd.read_csv(path)

# Drop missing values in crucial columns ('Reviews' and 'Rating')
df = df.dropna(subset=['Reviews', 'Rating'])

# Map 'Rating' to 3 sentiment classes: positive (>3), negative (<3), neutral (==3)
df['Sentiment'] = df['Rating'].apply(lambda r: 'positive' if r > 3 else ('negative' if r < 3 else 'neutral'))

# Function implementing 5 distinct text preprocessing steps
def preprocess_text(text):
    # Step 1: Convert text to lowercase
    text = text.lower()
    
    # Step 2: Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Step 3: Remove punctuation and special characters (keeping only letters and spaces)
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Step 4: Remove extra/multiple whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 5: Tokenization / Word splitting and joining
    words = text.split()
    
    return ' '.join(words)

# Apply preprocessing to the reviews
df['Cleaned_Reviews'] = df['Reviews'].astype(str).apply(preprocess_text)
print("Task 1 Complete. Sample cleaned review:")
print(df[['Reviews', 'Cleaned_Reviews', 'Sentiment']].head(2))

Task 1 Complete. Sample cleaned review:
                                             Reviews  \
0  I feel so LUCKY to have found this used (phone...   
1  nice phone, nice up grade from my pantach revu...   

                                     Cleaned_Reviews Sentiment  
0  i feel so lucky to have found this used phone ...  positive  
1  nice phone nice up grade from my pantach revue...  positive  


# Task 2: Use a Proper Train and Test Split

In [3]:
from sklearn.model_selection import train_test_split

X = df['Cleaned_Reviews']
y = df['Sentiment']

train_data, test_data, train_labels, test_labels = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Task 2 Complete.")
print(f"Training set size: {train_data.shape[0]}")
print(f"Testing set size: {test_data.shape[0]}")

Task 2 Complete.
Training set size: 331016
Testing set size: 82754


# Task 3: Do Feature Extraction Using TF-IDF

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
train_vectors = tfidf_vectorizer.fit_transform(train_data)
test_vectors = tfidf_vectorizer.transform(test_data)

print("Task 3 Complete.")
print(f"Train vectors shape: {train_vectors.shape}")
print(f"Test vectors shape: {test_vectors.shape}")

Task 3 Complete.
Train vectors shape: (331016, 5000)
Test vectors shape: (82754, 5000)


# Task 4: Train a Naive Bayes Classifier

In [5]:
from sklearn.naive_bayes import MultinomialNB

nb_classifier = MultinomialNB()
nb_classifier.fit(train_vectors, train_labels)

print("Task 4 Complete: Naive Bayes Classifier successfully trained.")

Task 4 Complete: Naive Bayes Classifier successfully trained.


# Task 5: Evaluate the Model on the Test Set

In [6]:
from sklearn.metrics import accuracy_score, classification_report

predictions = nb_classifier.predict(test_vectors)

accuracy = accuracy_score(test_labels, predictions)
print(f"Task 5 Complete.")
print(f"Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(test_labels, predictions))

Task 5 Complete.
Accuracy: 0.8436

Classification Report:
              precision    recall  f1-score   support

    negative       0.80      0.75      0.77     19412
     neutral       0.38      0.04      0.07      6352
    positive       0.86      0.97      0.91     56990

    accuracy                           0.84     82754
   macro avg       0.68      0.58      0.58     82754
weighted avg       0.81      0.84      0.81     82754



# Task 6: Print the Confusion Matrix

In [7]:
from sklearn.metrics import confusion_matrix

predictions = nb_classifier.predict(test_vectors)
conf_matrix = confusion_matrix(test_labels, predictions)

print("Task 6 Complete. Confusion Matrix:")
print(conf_matrix)

Task 6 Complete. Confusion Matrix:
[[14462   119  4831]
 [ 1984   231  4137]
 [ 1614   256 55120]]
